<a href="https://colab.research.google.com/github/aayush-jain-dtu/inventory-stock-prediction/blob/main/covid_new_dataset_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from datetime import date, timedelta
import random

In [ ]:
# =============================================================================
# STOCKIFY - MAJOR PROJECT DATASET (2014-2022)
# Engineered with COVID-19 Concept Drift for AML Adaptivity Proof
#
# DRIFT PHASES:
#   Phase 1 - PRE-COVID    (Jan 2014 – Dec 2019): Stable seasonal baselines
#   Phase 2 - COVID SHOCK  (Jan 2020 – Jun 2021): Massive demand reversal
#   Phase 3 - RECOVERY     (Jul 2021 – Dec 2022): Partial normalisation,
#                           permanent behavioral shifts retained
# =============================================================================

# -----------------------------------------------------------------------------
# PRODUCT CATALOGUE
# Chosen to maximise contrast across COVID phases:
#   - Home/WFH products  → demand SURGE during COVID
#   - Fashion / Travel   → demand CRASH during COVID
#   - Gym Equipment      → mixed (crash → surge as home-gym trend kicks in)
#   - Groceries/Hygiene  → surge (panic buying, hygiene awareness)
#   - Electronics        → surge (WFH, online school)
# -----------------------------------------------------------------------------
products = {
    # ── Electronics ──────────────────────────────────────────────────────────
    'P001': {
        'title': 'Laptop',
        'category': 'Electronics',
        'price_inr': 55000,
        'stock_range': (30, 150),
        'current_stock': 120,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'wfh_covid_surge',
        # Pre-COVID: back-to-school + festive; COVID: WFH explosion
    },
    'P002': {
        'title': 'Webcam & Headset Combo',
        'category': 'Electronics',
        'price_inr': 4500,
        'stock_range': (100, 600),
        'current_stock': 400,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'wfh_covid_surge',
    },
    'P003': {
        'title': 'Smartphone',
        'category': 'Electronics',
        'price_inr': 22000,
        'stock_range': (80, 400),
        'current_stock': 300,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'festive_and_new_year',
    },
    # ── Home & Kitchen ────────────────────────────────────────────────────────
    'P004': {
        'title': 'Air Purifier',
        'category': 'Home & Kitchen',
        'price_inr': 12000,
        'stock_range': (50, 300),
        'current_stock': 200,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'covid_hygiene_surge',
        # Spikes in winter + massive COVID surge (air quality awareness)
    },
    'P005': {
        'title': 'Microwave Oven',
        'category': 'Home & Kitchen',
        'price_inr': 9000,
        'stock_range': (60, 300),
        'current_stock': 220,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'covid_home_surge',
        # People cooking at home during lockdown
    },
    'P006': {
        'title': 'Office Chair (Ergonomic)',
        'category': 'Home & Kitchen',
        'price_inr': 8500,
        'stock_range': (40, 200),
        'current_stock': 150,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'wfh_covid_surge',
    },
    # ── Gym Equipment ─────────────────────────────────────────────────────────
    'P007': {
        'title': 'Treadmill',
        'category': 'Gym Equipment',
        'price_inr': 42000,
        'stock_range': (10, 80),
        'current_stock': 60,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'jan_spike_then_home_gym',
        # Jan New Year spike pre-COVID; gyms closed → home-gym surge mid-2020
    },
    'P008': {
        'title': 'Resistance Bands & Dumbbell Set',
        'category': 'Gym Equipment',
        'price_inr': 3500,
        'stock_range': (200, 1200),
        'current_stock': 900,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'jan_spike_then_home_gym',
    },
    'P009': {
        'title': 'Yoga Mat',
        'category': 'Gym Equipment',
        'price_inr': 1500,
        'stock_range': (300, 1500),
        'current_stock': 1200,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'jan_spike_then_home_gym',
    },
    # ── Fashion ───────────────────────────────────────────────────────────────
    'P010': {
        'title': "Men's Formal Shirts",
        'category': 'Fashion',
        'price_inr': 1800,
        'stock_range': (400, 2000),
        'current_stock': 1500,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'covid_crash_formal',
        # Pre-COVID: festive/office; COVID: near-zero (no office, no events)
    },
    'P011': {
        'title': 'Casual Loungewear Set',
        'category': 'Fashion',
        'price_inr': 1200,
        'stock_range': (500, 2500),
        'current_stock': 2000,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'covid_lounge_surge',
        # Inverse of formal — surges during lockdown
    },
    'P012': {
        'title': 'Festive Sarees',
        'category': 'Fashion',
        'price_inr': 6000,
        'stock_range': (300, 1500),
        'current_stock': 1200,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'festive_wedding_covid_crash',
        # Strong Oct-Dec + Feb-Mar pre-COVID; crashes with event bans
    },
    # ── Groceries & Hygiene ───────────────────────────────────────────────────
    'P013': {
        'title': 'Hand Sanitizer (Bulk Pack)',
        'category': 'Groceries & Hygiene',
        'price_inr': 1200,
        'stock_range': (300, 2000),
        'current_stock': 1000,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'covid_hygiene_surge',
        # Practically zero pre-COVID; panic-buying surge; normalises post-COVID
    },
    'P014': {
        'title': 'Packaged Grocery Staples',
        'category': 'Groceries & Hygiene',
        'price_inr': 800,
        'stock_range': (500, 3000),
        'current_stock': 2500,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'steady_covid_spike',
        # Always in demand; huge panic-buying spike in early COVID
    },
    # ── Stationery & EdTech ───────────────────────────────────────────────────
    'P015': {
        'title': 'Drawing Tablet (EdTech)',
        'category': 'Stationery & EdTech',
        'price_inr': 7500,
        'stock_range': (50, 300),
        'current_stock': 200,
        'last_restock': date(2014, 1, 1),
        'demand_pattern': 'edtech_covid_surge',
        # Online school surge during COVID
    },
}

In [ ]:
# -----------------------------------------------------------------------------
# CLIENT POOL
# -----------------------------------------------------------------------------
clients = [f'C{str(i).zfill(3)}' for i in range(1, 21)]  # 20 clients

# -----------------------------------------------------------------------------
# DATE RANGE  &  PHASE HELPERS
# -----------------------------------------------------------------------------
start_date = date(2014, 1, 1)
end_date   = date(2022, 12, 31)
date_list  = [start_date + timedelta(days=x)
              for x in range((end_date - start_date).days + 1)]

def get_phase(d: date) -> str:
    """Return the demand-regime phase for a given date."""
    if d < date(2020, 3, 1):
        return 'pre_covid'
    elif d < date(2021, 7, 1):
        return 'covid'
    else:
        return 'recovery'

In [ ]:
# -----------------------------------------------------------------------------
# RESTOCK INTERVALS (days between scheduled restocks)
# -----------------------------------------------------------------------------
restock_intervals = {pid: random.randint(25, 55) for pid in products}


In [ ]:
# =============================================================================
# MAIN SIMULATION LOOP
# =============================================================================
data = []
order_id_counter = 1
rows_generated   = 0
TARGET_ROWS      = 8000   # larger dataset = more convincing backtesting

while rows_generated < TARGET_ROWS:
    order_date  = random.choice(date_list)
    order_year  = order_date.year
    order_month = order_date.month
    order_day   = order_date.day
    phase       = get_phase(order_date)

    # ── STEP 1: Build product weights ────────────────────────────────────────
    weights = {pid: 1 for pid in products}

    # ── PRE-COVID baselines (seasonal) ───────────────────────────────────────
    if phase == 'pre_covid':
        # Electronics
        if order_month in [7, 8]:            # back-to-school
            weights['P001'] *= 2
            weights['P015'] *= 3
        if order_month in [10, 11, 12]:      # festive season
            weights['P001'] *= 2; weights['P002'] *= 2; weights['P003'] *= 3
            weights['P010'] *= 2; weights['P012'] *= 3
        if order_month == 1:                 # New Year
            weights['P007'] *= 3; weights['P008'] *= 3; weights['P009'] *= 3
            weights['P003'] *= 2
        if order_month in [2, 3]:            # wedding season
            weights['P012'] *= 2; weights['P010'] *= 2
        if order_month in [5, 6, 7]:         # summer
            weights['P005'] *= 2             # cooking at home (summer holidays)

    # ── COVID SHOCK (demand reversal) ────────────────────────────────────────
    elif phase == 'covid':
        # WFH surge
        weights['P001'] *= 5   # laptops
        weights['P002'] *= 6   # webcam/headset
        weights['P006'] *= 5   # office chairs

        # Home-gym surge (gyms closed from March 2020)
        weights['P007'] *= 4
        weights['P008'] *= 5
        weights['P009'] *= 5

        # Hygiene panic
        weights['P013'] *= 8
        weights['P014'] *= 4

        # Home cooking
        weights['P005'] *= 4
        weights['P004'] *= 3   # air purifier

        # EdTech (online school)
        weights['P015'] *= 5

        # Loungewear replaces formal
        weights['P011'] *= 5

        # CRASH products (events/offices closed)
        weights['P010'] *= 0  # formal shirts → near zero
        weights['P012'] *= 0  # festive sarees → events banned

        # Slight festive recovery in Oct-Nov 2020 (unlock phases)
        if order_month in [10, 11] and order_year == 2020:
            weights['P003'] *= 2
            weights['P012']  = 1   # small recovery

    # ── RECOVERY (partial normalisation, some shifts permanent) ──────────────
    else:  # recovery
        # WFH partially persists (hybrid work)
        weights['P001'] *= 2
        weights['P002'] *= 2
        weights['P006'] *= 2

        # Home-gym culture sticks
        weights['P007'] *= 2
        weights['P008'] *= 3
        weights['P009'] *= 3

        # Hygiene normalises (not panic, but elevated)
        weights['P013'] *= 2

        # Fashion recovery
        if order_month in [10, 11, 12]:
            weights['P010'] *= 2
            weights['P012'] *= 2

        # Seasonal baselines return
        if order_month == 1:
            weights['P007'] *= 2; weights['P008'] *= 2; weights['P009'] *= 2
        if order_month in [7, 8]:
            weights['P015'] *= 2

    # ── STEP 2: Select product ───────────────────────────────────────────────
    pid_list = list(products.keys())
    w_list   = [max(weights[p], 0) for p in pid_list]   # no negative weights

    # Guard against all-zero weights (extremely unlikely but safe)
    if sum(w_list) == 0:
        continue

    selected_pid  = random.choices(pid_list, weights=w_list, k=1)[0]
    product_info  = products[selected_pid]

    # ── STEP 3: Restock logic ────────────────────────────────────────────────
    safety_stock = product_info['stock_range'][0] * 0.2
    if product_info['current_stock'] <= safety_stock:
        restock_qty = product_info['stock_range'][1] - product_info['current_stock']
        product_info['current_stock'] += restock_qty

    if (order_date - product_info['last_restock']).days >= restock_intervals[selected_pid]:
        restock_qty = random.randint(
            product_info['stock_range'][0] // 2,
            product_info['stock_range'][0]
        )
        product_info['current_stock'] += restock_qty
        product_info['last_restock']   = order_date

    # ── STEP 4: Quantity ordered ─────────────────────────────────────────────
    effective_weight = weights[selected_pid]

    # COVID era: scarce items like sanitizer ordered in bulk
    if phase == 'covid' and selected_pid == 'P013':
        quantity_ordered = random.randint(20, 60)

    elif effective_weight >= 4:         # high-demand season
        if product_info['price_inr'] > 20000:
            quantity_ordered = random.randint(5, 15)
        else:
            quantity_ordered = random.randint(15, 50)

    elif effective_weight >= 2:         # moderate season
        if product_info['price_inr'] > 20000:
            quantity_ordered = random.randint(2, 8)
        else:
            quantity_ordered = random.randint(5, 20)

    else:                               # off-season / baseline
        if product_info['price_inr'] > 20000:
            quantity_ordered = random.randint(1, 3)
        else:
            quantity_ordered = random.randint(1, 6)

    # ── STEP 5: Simulated annual price inflation (~3% per year) ──────────────
    years_passed = order_year - start_date.year
    price = int(product_info['price_inr'] * (1.03 ** years_passed))

    # COVID supply-chain shock: price spike for high-demand items (2020-2021)
    if phase == 'covid' and effective_weight >= 4:
        price = int(price * random.uniform(1.10, 1.35))   # 10-35% premium

    # ── STEP 6: Enforce stock constraint ────────────────────────────────────
    quantity_ordered = min(quantity_ordered, product_info['current_stock'])
    if quantity_ordered == 0:
        continue

    # ── STEP 7: Record row ───────────────────────────────────────────────────
    data.append([
        f'O{str(order_id_counter).zfill(5)}',
        order_year,
        order_month,
        order_day,
        phase,                              # NEW: explicit phase label
        selected_pid,
        product_info['title'],
        product_info['category'],
        product_info['current_stock'],
        random.choice(clients),
        price,
        quantity_ordered,
    ])

    product_info['current_stock'] -= quantity_ordered
    order_id_counter += 1
    rows_generated   += 1

# =============================================================================
# BUILD DATAFRAME & SAVE
# =============================================================================
columns = [
    'order_id', 'order_year', 'order_month', 'order_day',
    'demand_phase',             # pre_covid / covid / recovery
    'product_id', 'product_title', 'product_category',
    'current_product_stock', 'client_id', 'price_inr', 'quantity_ordered'
]

df = pd.DataFrame(data, columns=columns)
df.sort_values(by=['order_year', 'order_month', 'order_day'], inplace=True)
df.reset_index(drop=True, inplace=True)

output_path = 'inventory_dataset_covid_drift.csv'
df.to_csv(output_path, index=False)

# =============================================================================
# QUICK VALIDATION SUMMARY
# =============================================================================
print("=" * 60)
print("  STOCKIFY – COVID DRIFT DATASET GENERATION COMPLETE")
print("=" * 60)
print(f"\n  Total rows      : {len(df):,}")
print(f"  Date range      : {df['order_year'].min()} – {df['order_year'].max()}")
print(f"  Unique products : {df['product_id'].nunique()}")
print(f"  Unique clients  : {df['client_id'].nunique()}")

print("\n  Rows by demand phase:")
phase_counts = df['demand_phase'].value_counts()
for phase, count in phase_counts.items():
    print(f"    {phase:<15} : {count:,}")

print("\n  Average quantity_ordered by phase:")
phase_avg = df.groupby('demand_phase')['quantity_ordered'].mean().round(2)
for phase, avg in phase_avg.items():
    print(f"    {phase:<15} : {avg}")

print("\n  Top products by volume (COVID phase):")
covid_df = df[df['demand_phase'] == 'covid']
top_covid = (
    covid_df.groupby('product_title')['quantity_ordered']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
for prod, vol in top_covid.items():
    print(f"    {prod:<35} : {vol:,}")

print("\n  Top products by volume (Pre-COVID phase):")
pre_df = df[df['demand_phase'] == 'pre_covid']
top_pre = (
    pre_df.groupby('product_title')['quantity_ordered']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
for prod, vol in top_pre.items():
    print(f"    {prod:<35} : {vol:,}")

print(f"\n  Saved to: {output_path}")
print("=" * 60)


  STOCKIFY – COVID DRIFT DATASET GENERATION COMPLETE

  Total rows      : 8,000
  Date range      : 2014 – 2022
  Unique products : 15
  Unique clients  : 20

  Rows by demand phase:
    pre_covid       : 5,532
    recovery        : 1,308
    covid           : 1,160

  Average quantity_ordered by phase:
    covid           : 27.61
    pre_covid       : 5.32
    recovery        : 9.26

  Top products by volume (COVID phase):
    Hand Sanitizer (Bulk Pack)          : 5,919
    Webcam & Headset Combo              : 3,984
    Yoga Mat                            : 3,454
    Resistance Bands & Dumbbell Set     : 3,221
    Drawing Tablet (EdTech)             : 2,854

  Top products by volume (Pre-COVID phase):
    Festive Sarees                      : 4,196
    Men's Formal Shirts                 : 4,025
    Microwave Oven                      : 2,990
    Drawing Tablet (EdTech)             : 2,691
    Webcam & Headset Combo              : 2,445

  Saved to: inventory_dataset_covid_drift.csv
